# Intro to Web Scraping

Web scraping is the process of extracting data from websites. It can be a helpful tool for gathering information that is not readily available through APIs or other structured data sources.

How you scrape depends a lot on how information is contained in a website. Older and less interactive pages that provide data immediately in the initial HTML are typically easier to scrape than JavaScript-heavy pages require a browser-like environment to execute code and reveal the data. 

This lesson will give a brief example of both types of scraping.

## Simple HTML pages

Older websites that are not very interactive often have all the information you need contained in the initial HTML. This makes them relatively easy to scrape using libraries like BeautifulSoup in Python. You can simply send a request to the webpage, get the HTML content, and then parse it to extract the data you need.

Some tips to extracting the right HTML when co-programming with an LLM:
* Share the link with your LLM.
* Open up the "Inspect" tool in your browser to see the HTML structure of the page. This will help you identify the tags and classes that contain the data you want to scrape.
* Copy relevant parts of the HTML and share it with your LLM. (You can also download the entire webpage as an HTML and share the file directly.)

In [2]:
import re
import warnings

import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib3.exceptions import NotOpenSSLWarning

warnings.filterwarnings("ignore", category=NotOpenSSLWarning)

URL = "https://www.scotusblog.com/case-files/terms/ot2025/"


def clean_text(value):
    return re.sub(r"\s+", " ", value or "").strip()


def parse_case_cell(case_cell):
    heading = case_cell.find_previous("h2")
    section = clean_text(heading.get_text(" ", strip=True)) if heading else None

    top_div = case_cell.find("div", recursive=False)
    summary_div = case_cell.find("div", class_=lambda value: value and "leading-[26px]" in value)
    links_div = case_cell.find("div", class_="links")

    case_link = None
    for link in top_div.find_all("a", href=True):
        href = link["href"]
        if "/cases/case-files/" in href:
            case_link = link
            break

    docket_link = None
    for link in top_div.find_all("a", href=True):
        text = clean_text(link.get_text(" ", strip=True))
        if re.fullmatch(r"\d+[A-Z-]*-\d+[A-Z-]*", text) or re.fullmatch(r"\d+[A-Z]?\d*", text):
            docket_link = link
            break

    metadata_text = clean_text(top_div.get_text(" ", strip=True))
    argument_date_match = re.search(r"\[Arg:\s*([0-9.]+)", metadata_text)
    decided_date_match = re.search(r";\s*Decided\s*([0-9.]+)", metadata_text)

    transcript_url = None
    audio_url = None
    decided_url = None
    for link in top_div.find_all("a", href=True):
        label = clean_text(link.get_text(" ", strip=True))
        href = link["href"]
        if label == "Transcript":
            transcript_url = href
        elif label == "Audio":
            audio_url = href
        elif decided_date_match and label == decided_date_match.group(1):
            decided_url = href

    summary_text = clean_text(summary_div.get_text(" ", strip=True)) if summary_div else ""
    summary_match = re.match(r"^(Holding|Issue\(s\)):\s*(.*)$", summary_text)
    summary_label = summary_match.group(1) if summary_match else None
    summary_value = summary_match.group(2) if summary_match else summary_text

    related_links = []
    if links_div:
        for link in links_div.find_all("a", href=True):
            label = clean_text(link.get_text(" ", strip=True))
            related_links.append({"label": label, "url": link["href"]})

    related_link_lookup = {item["label"]: item["url"] for item in related_links}

    return {
        "section": section,
        "case_name": clean_text(case_link.get_text(" ", strip=True)) if case_link else None,
        "case_url": case_link["href"] if case_link else None,
        "docket_number": clean_text(docket_link.get_text(" ", strip=True)) if docket_link else None,
        "docket_url": docket_link["href"] if docket_link else None,
        "argument_date": argument_date_match.group(1) if argument_date_match else None,
        "transcript_url": transcript_url,
        "audio_url": audio_url,
        "decided_date": decided_date_match.group(1) if decided_date_match else None,
        "decided_url": decided_url,
        "summary_type": summary_label,
        "summary_text": summary_value,
        "case_preview_url": related_link_lookup.get("Case Preview"),
        "argument_analysis_url": related_link_lookup.get("Argument Analysis"),
        "opinion_analysis_url": related_link_lookup.get("Opinion Analysis"),
        "related_links": related_links,
    }


response = requests.get(
    URL,
    headers={
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"
    },
    timeout=30,
    )
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")
case_cells = soup.select("table.caseindex td.py-4.border-b.border-gray-400")
scotus_df = pd.DataFrame(parse_case_cell(case_cell) for case_cell in case_cells)

print(f"Collected {len(scotus_df)} cases")
scotus_df.head(10)

Collected 64 cases


,section,case_name,case_url,docket_number,docket_url,argument_date,transcript_url,audio_url,decided_date,decided_url,summary_type,summary_text,case_preview_url,argument_analysis_url,opinion_analysis_url,related_links
0,October Sitting (10),Villarreal v. Texas,https://www.scotusblog.com/cases/case-files/vi...,24-557,https://www.supremecourt.gov/docket/docketfile...,10.06.2025,https://www.supremecourt.gov/oral_arguments/ar...,https://www.supremecourt.gov/oral_arguments/au...,02.25.2026,https://www.supremecourt.gov/opinions/25pdf/24...,Holding,A trial court’s qualified conferral order that...,https://www.scotusblog.com/2025/10/supreme-cou...,https://www.scotusblog.com/2025/10/court-leans...,https://www.scotusblog.com/2026/02/court-rules...,"[{'label': 'Case Preview', 'url': 'https://www..."
1,October Sitting (10),Berk v. Choy,https://www.scotusblog.com/cases/case-files/be...,24-440,https://www.supremecourt.gov/docket/docketfile...,10.06.2025,https://www.supremecourt.gov/oral_arguments/ar...,https://www.supremecourt.gov/oral_arguments/au...,01.20.2026,https://www.supremecourt.gov/opinions/25pdf/24...,Holding,Delaware law requiring a plaintiff suing for m...,https://www.scotusblog.com/2025/10/do-state-li...,https://www.scotusblog.com/2025/10/justices-de...,https://www.scotusblog.com/2026/01/justices-re...,"[{'label': 'Case Preview', 'url': 'https://www..."
2,October Sitting (10),Barrett v. U.S.,https://www.scotusblog.com/cases/case-files/ba...,24-5774,https://www.supremecourt.gov/docket/docketfile...,10.07.2025,https://www.supremecourt.gov/oral_arguments/ar...,https://www.supremecourt.gov/oral_arguments/au...,01.14.2026,https://www.supremecourt.gov/opinions/25pdf/24...,Holding,Congress did not clearly authorize convictions...,https://www.scotusblog.com/2025/10/justices-to...,https://www.scotusblog.com/2025/10/court-consi...,https://www.scotusblog.com/2026/01/court-unani...,"[{'label': 'Case Preview', 'url': 'https://www..."
3,October Sitting (10),Chiles v. Salazar (Conversion Therapy),https://www.scotusblog.com/cases/case-files/ch...,24-539,https://www.supremecourt.gov/docket/docketfile...,10.07.2025,https://www.supremecourt.gov/oral_arguments/ar...,https://www.supremecourt.gov/oral_arguments/au...,03.31.2026,https://www.supremecourt.gov/opinions/25pdf/24...,Holding,"Colorado’s law banning conversion therapy, as ...",https://www.scotusblog.com/2025/10/does-colora...,https://www.scotusblog.com/2025/10/majority-of...,https://www.scotusblog.com/2026/03/supreme-cou...,"[{'label': 'Case Preview', 'url': 'https://www..."
4,October Sitting (10),Bost v. Illinois State Board of Elections,https://www.scotusblog.com/cases/case-files/bo...,24-568,https://www.supremecourt.gov/docket/docketfile...,10.08.2025,https://www.supremecourt.gov/oral_arguments/ar...,https://www.supremecourt.gov/oral_arguments/au...,01.14.2026,https://www.supremecourt.gov/opinions/25pdf/24...,Holding,"As a candidate for office, Congressman Michael...",https://www.scotusblog.com/2025/10/when-may-a-...,https://www.scotusblog.com/2025/10/justices-se...,https://www.scotusblog.com/2026/01/court-holds...,"[{'label': 'Case Preview', 'url': 'https://www..."
5,October Sitting (10),U.S. Postal Service v. Konan,https://www.scotusblog.com/cases/case-files/un...,24-351,https://www.supremecourt.gov/docket/docketfile...,10.08.2025,https://www.supremecourt.gov/oral_arguments/ar...,https://www.supremecourt.gov/oral_arguments/au...,02.24.2026,https://www.supremecourt.gov/opinions/25pdf/24...,Holding,The United States retains sovereign immunity f...,https://www.scotusblog.com/2025/10/how-a-mail-...,https://www.scotusblog.com/2025/10/court-debat...,https://www.scotusblog.com/2026/02/court-holds...,"[{'label': 'Case Preview', 'url': 'https://www..."
6,October Sitting (10),Bowe v. U.S.,https://www.scotusblog.com/cases/case-files/bo...,24-5438,https://www.supremecourt.gov/docket/docketfile...,10.14.2025,https://www.supremecourt.gov/oral_arguments/ar...,https://www.supremecourt.gov/oral_argument

In [4]:
scotus_df.sample(3)

,section,case_name,case_url,docket_number,docket_url,argument_date,transcript_url,audio_url,decided_date,decided_url,summary_type,summary_text,case_preview_url,argument_analysis_url,opinion_analysis_url,related_links
41,March Sitting (8),Watson v. Republican National Committee (Elect...,https://www.scotusblog.com/cases/case-files/wa...,24-1260,https://www.supremecourt.gov/docket/docketfile...,03.23.2026,https://www.supremecourt.gov/oral_arguments/ar...,https://www.supremecourt.gov/oral_arguments/au...,None,None,Issue(s),"Whether the federal election-day statutes, 2 U...",https://www.scotusblog.com/2026/03/court-to-he...,https://www.scotusblog.com/2026/03/court-appea...,None,"[{'label': 'Case Preview', 'url': 'https://www..."
7,October Sitting (10),Ellingburg v. U.S.,https://www.scotusblog.com/cases/case-files/el...,24-482,https://www.supremecourt.gov/docket/docketfile...,10.14.2025,https://www.supremecourt.gov/oral_arguments/ar...,https://www.supremecourt.gov/oral_arguments/au...,01.20.2026,https://www.supremecourt.gov/opinions/25pdf/24...,Holding,Restitution under the Mandatory Victims Restit...,https://www.scotusblog.com/2025/10/court-to-co...,https://www.scotusblog.com/2025/10/justices-de...,https://www.scotusblog.com/2026/01/justices-ho...,"[{'label': 'Case Preview', 'url': 'https://www..."
62,Cases decided without oral argument (5),Clark v. Sweeney,https://www.scotusblog.com/cases/case-files/cl...,25-52,https://www.supremecourt.gov/docket/docketfile...,None,None,None,None,None,Holding,The U.S. Court of Appeals for the 4th Circuit ...,None,None,None,[]


If you wanted to make this more sophisticated, you could have your scraper click through the different menu options and scrape each page.

## JavaScript-heavy pages

Examle: Remember that Audobon example from week 5, where we practiced creating dictionaries on a csv of the National Audobon Society's [Guide to North American Birds](https://www.audubon.org/bird-guide?sort_by=name&mode=detail)? I used Copilot to scrape that information from Audobon's website.

The biggest challenge to scraping this site is that all the birds are not shown on the page at once. You have to scroll to be able to see all the birds and scrape as you go, which means your scraper needs to be able to do that autonomously. 

You can use an external package like Selenium (my personal favorite) or Playwright to do this. These packages allow you to create a browser-like environment that can execute JavaScript and interact with the page as a user would, including scrolling.

These packages can also deal with captchas, which are often used to prevent automated scraping. I have found them to be only somewhat successful at this, but they can still be a useful tool in your scraping toolkit.

In [6]:
import re
import time

import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait

AUDUBON_URL = "https://www.audubon.org/bird-guide?family=blackbird&sort_by=name&mode=detail"
TARGET_BIRDS = 20
SCROLL_PAUSE_SECONDS = 1.2


def clean_text(value):
    return re.sub(r"\s+", " ", value or "").strip()


def count_loaded_cards(driver):
    return len(driver.find_elements(By.CSS_SELECTOR, "div.bird-card-2023 .bird-card-title"))


def dismiss_cookie_banner(driver):
    button_selectors = [
        "button#onetrust-accept-btn-handler",
        "button[aria-label='Accept']",
        "button[title='Accept']",
    ]

    for selector in button_selectors:
        buttons = driver.find_elements(By.CSS_SELECTOR, selector)
        if buttons:
            try:
                driver.execute_script("arguments[0].click();", buttons[0])
                time.sleep(1)
                return
            except Exception:
                pass

    xpath_buttons = driver.find_elements(
        By.XPATH,
        "//button[contains(translate(normalize-space(.), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'accept')]",
    )
    if xpath_buttons:
        try:
            driver.execute_script("arguments[0].click();", xpath_buttons[0])
            time.sleep(1)
        except Exception:
            pass


def scroll_until_cards_loaded(driver, target_count):
    loaded_count = count_loaded_cards(driver)
    stall_count = 0
    print(f"Initially loaded: {loaded_count} birds")

    while loaded_count < target_count and stall_count < 5:
        titles = driver.find_elements(By.CSS_SELECTOR, "div.bird-card-2023 .bird-card-title")
        if not titles:
            break

        driver.execute_script(
            "arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});",
            titles[-1],
        )
        time.sleep(SCROLL_PAUSE_SECONDS)
        driver.execute_script("window.scrollBy(0, Math.floor(window.innerHeight * 0.9));")

        try:
            WebDriverWait(driver, 5).until(
                lambda current_driver: count_loaded_cards(current_driver) > loaded_count
            )
            new_count = count_loaded_cards(driver)
            print(f"Loaded {new_count} birds")
            loaded_count = new_count
            stall_count = 0
        except TimeoutException:
            stall_count += 1
            loaded_count = count_loaded_cards(driver)
            print(f"Scroll retry {stall_count}: still at {loaded_count} birds")
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(SCROLL_PAUSE_SECONDS)

    return min(loaded_count, target_count)


def parse_bird_card(card):
    bird_link = card.select_one("a.card-link")
    name = card.select_one(".bird-card-title")
    conservation_status = card.select_one(".bird-card-status .detail-content")
    habitat = card.select_one(".bird-card-habitats .detail-content")
    summary = card.select_one(".bird-card-description .detail-content")

    return {
        "bird_name": clean_text(name.get_text(" ", strip=True)) if name else None,
        "conservation_status": clean_text(conservation_status.get_text(" ", strip=True)) if conservation_status else None,
        "habitat": clean_text(habitat.get_text(" ", strip=True)) if habitat else None,
        "summary": clean_text(summary.get_text(" ", strip=True)) if summary else None,
        "bird_url": bird_link.get("href") if bird_link else None,
    }


chrome_options = webdriver.ChromeOptions()
# Keep the browser visible so you can watch the scrolling demo.
chrome_options.add_argument("--window-size=1440,1100")
chrome_options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Chrome(options=chrome_options)

try:
    print(f"Opening {AUDUBON_URL}")
    driver.get(AUDUBON_URL)

    WebDriverWait(driver, 30).until(
        lambda current_driver: len(current_driver.find_elements(By.CSS_SELECTOR, "div.bird-card-2023")) > 0
    )
    dismiss_cookie_banner(driver)

    if count_loaded_cards(driver) == 0:
        search_buttons = driver.find_elements(By.CSS_SELECTOR, "#edit-submit-nas-bird-guide")
        if search_buttons:
            driver.execute_script("arguments[0].click();", search_buttons[0])
            time.sleep(2)

    WebDriverWait(driver, 30).until(
        lambda current_driver: count_loaded_cards(current_driver) > 0
    )

    collected_count = scroll_until_cards_loaded(driver, TARGET_BIRDS)

    soup = BeautifulSoup(driver.page_source, "html.parser")
    bird_cards = [
        card for card in soup.select("div.bird-card-2023")
        if card.select_one(".bird-card-title")
    ][:TARGET_BIRDS]
    audubon_df = pd.DataFrame(parse_bird_card(card) for card in bird_cards)

    print(f"Collected {len(audubon_df)} birds after scrolling")
    time.sleep(2)
finally:
    driver.quit()

audubon_df

Opening https://www.audubon.org/bird-guide?family=blackbird&sort_by=name&mode=detail
Initially loaded: 8 birds
Loaded 16 birds
Loaded 24 birds
Collected 20 birds after scrolling


,bird_name,conservation_status,habitat,summary,bird_url
0,Altamira Oriole,Least Concern,Forests and Woodlands,The Altamira Oriole is common in northeastern ...,/field-guide/bird/altamira-oriole
1,Audubon's Oriole,Least Concern,"Forests and Woodlands, Shrublands, Savannas, a...",In native woodlands and brushy country of far ...,/field-guide/bird/audubons-oriole
2,Baltimore Oriole,Least Concern,"Fields, Meadows, and Grasslands, Forests and W...",One of the most brilliantly colored songbirds ...,/field-guide/bird/baltimore-oriole
3,Black-vented Oriole,Least Concern,"Forests and Woodlands, Shrublands, Savannas, a...","In Mexico and Central America, this large orio...",/field-guide/bird/black-vented-oriole
4,Boat-tailed Grackle,Least Concern,"Coasts and Shorelines, Fields, Meadows, and Gr...","Until the 1970s, this big blackbird was consid...",/field-guide/bird/boat-tailed-grackle
5,Bobolink,Least Concern,"Fields, Meadows, and Grasslands, Freshwater We...",Fluttering over meadows and hayfields in summe...,/field-guide/bird/bobolink
6,Brewer's Blackbird,Least Concern,"Coasts and Shorelines, Desert and Arid Habitat...",This is the common blackbird of open country i...,/field-guide/bird/brewers-blackbird
7,Bronzed Cowbird,Least Concern,"Arroyos and Canyons, Desert and Arid Habitats,...",Larger than the Brown-headed Cowbird and mostl...,/field-guide/bird/bronzed-cowbird
8,Brown-headed Cowbird,Least Concern,"Arroyos and Canyons, Desert and Arid Habitats,...","Centuries ago, the Brown-headed Cowbird probab...",/field-guide/bird/brown-headed-cowbird
9,Bullock's Oriole,Least Concern,"Forests and Woodlands, Shrublands, Savannas, a...","In the west, the Bullock's Oriole is common in...",/field-guide/bird/bullocks-oriole


In [7]:
# another example: pre-selecting options on NCSL

import os
import re
import time
from datetime import datetime
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.select import Select
from bs4 import BeautifulSoup

# ============================================================================
# CONFIGURATION
# ============================================================================

SELECTED_STATE = os.getenv('NCSL_STATE', 'New Jersey')
USE_ARCHIVE = os.getenv('NCSL_USE_ARCHIVE', 'false').lower() == 'true'

# Set URL based on archive flag
if USE_ARCHIVE:
    NCSL_URL = "https://www.ncsl.org/elections-and-campaigns/state-elections-legislation-archived-database"
else:
    NCSL_URL = "https://www.ncsl.org/elections-and-campaigns/state-election-legislation-database"

# Parse years from environment variable (comma-separated for multiple years)
CURRENT_YEAR = datetime.now().year
YEARS_INPUT = os.getenv('NCSL_YEARS', str(CURRENT_YEAR))


def parse_years_input(years_input):
    """Parse a year string supporting comma-separated years and inclusive ranges."""
    parsed_years = []

    for part in years_input.split(','):
        token = part.strip()
        if not token:
            continue

        range_match = re.fullmatch(r'(\d{4})\s*-\s*(\d{4})', token)
        if range_match:
            start_year = int(range_match.group(1))
            end_year = int(range_match.group(2))
            step = 1 if start_year <= end_year else -1
            parsed_years.extend(range(start_year, end_year + step, step))
            continue

        if token.isdigit():
            parsed_years.append(int(token))

    ordered_unique_years = []
    seen_years = set()
    for year in parsed_years:
        if year not in seen_years:
            ordered_unique_years.append(year)
            seen_years.add(year)

    return ordered_unique_years


SELECTED_YEARS = parse_years_input(YEARS_INPUT)
if not SELECTED_YEARS:
    SELECTED_YEARS = [CURRENT_YEAR]


def normalize_topic_text(value):
    """Normalize topic text for matching against selector labels."""
    if not value:
        return ""
    text = str(value).replace('\xa0', ' ').replace('&nbsp;', ' ')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def extract_master_topics_from_page(page_source):
    """Build a code->display label mapping from the Topics selector table."""
    soup = BeautifulSoup(page_source, 'html.parser')
    topics_lookup = {}

    topics_table = soup.find('table', {'id': re.compile(r'StateNetDB_ckBxTopics$')})
    if not topics_table:
        return topics_lookup

    for row in topics_table.find_all('tr'):
        checkbox = row.find('input', {'type': 'checkbox'})
        label = row.find('label')
        if not checkbox or not label:
            continue

        topic_code = normalize_topic_text(checkbox.get('value', ''))
        topic_display = normalize_topic_text(label.get_text(' ', strip=True))
        if topic_code and topic_display and topic_display.lower() != 'all topics':
            topics_lookup[topic_code] = topic_display

    return topics_lookup


def parse_bill_topics(topics_text, master_topics):
    """Split raw bill topics by matching against canonical topic selector labels."""
    normalized_topics = normalize_topic_text(topics_text)
    if not normalized_topics or normalized_topics == 'N/A':
        return 'N/A'

    if not master_topics:
        return normalized_topics

    matched_topics = []
    lowered_topics = normalized_topics.lower()

    for display_name in master_topics.values():
        lowered_display = display_name.lower()
        position = lowered_topics.find(lowered_display)
        if position != -1:
            matched_topics.append((position, display_name))

    if not matched_topics:
        return normalized_topics

    matched_topics.sort(key=lambda item: (item[0], -len(item[1])))
    ordered_unique_topics = []
    seen = set()
    for _, display_name in matched_topics:
        if display_name not in seen:
            ordered_unique_topics.append(display_name)
            seen.add(display_name)

    return '; '.join(ordered_unique_topics)


def validate_config():
    """Validate and print run config."""
    database_type = "Archived" if USE_ARCHIVE else "Current"
    years_str = ", ".join(map(str, SELECTED_YEARS))
    print(f"✓ Configuration: {SELECTED_STATE} - Years: {years_str} ({database_type} Database)")


def scrape_bills(year=None):
    """Scrape bills from NCSL database for a selected year."""
    if year is None:
        year = SELECTED_YEARS[0]

    def select_year_if_available(driver, target_year):
        """Try to select a year from any dropdown that contains it."""
        year_str = str(target_year)
        selects = []
        try:
            year_select = driver.find_element(
                By.XPATH,
                "//div[contains(normalize-space(.), 'Year')]/following::select[1]",
            )
            selects.append(year_select)
        except Exception:
            pass

        selects.extend(driver.find_elements(By.TAG_NAME, "select"))
        for sel in selects:
            try:
                select = Select(sel)
                options = [opt.get_attribute("value") or opt.text for opt in select.options]
                if any(year_str == (opt or "").strip() for opt in options):
                    try:
                        select.select_by_value(year_str)
                    except Exception:
                        select.select_by_visible_text(year_str)
                    return True
            except Exception:
                continue
        return False

    print("\n" + "="*80)
    database_type = "ARCHIVED DATABASE" if USE_ARCHIVE else "CURRENT DATABASE"
    print(f"SCRAPING BILLS FROM NCSL {database_type} - YEAR {year}")
    print("="*80)

    chrome_options = Options()
    # Keep browser visible for class demo.
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")

    driver = webdriver.Chrome(options=chrome_options)
    bills_data = []

    try:
        print(f"Navigating to {NCSL_URL}")
        driver.get(NCSL_URL)
        time.sleep(2)

        # Close cookie banner when present.
        try:
            cookie_buttons = driver.find_elements(By.XPATH, "//button[contains(text(), 'Accept')] | //button[contains(text(), 'accept')]")
            if not cookie_buttons:
                cookie_buttons = driver.find_elements(By.XPATH, "//button[@class*='cookie'] | //button[@class*='consent']")
            if cookie_buttons:
                cookie_buttons[0].click()
                time.sleep(1)
        except Exception:
            pass

        state_label = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, f"//label[contains(text(), '{SELECTED_STATE}')]"))
        )
        driver.execute_script("arguments[0].click();", state_label)
        print(f"✓ Selected {SELECTED_STATE}")
        time.sleep(1)

        try:
            if select_year_if_available(driver, year):
                print(f"✓ Selected year {year}")
                time.sleep(1)
            else:
                print(f"⚠ Year dropdown not found for {year}; using site default")
        except Exception as e:
            print(f"⚠ Could not select year {year}: {str(e)}")

        master_topics = extract_master_topics_from_page(driver.page_source)
        if master_topics:
            print(f"✓ Loaded {len(master_topics)} master topics from selectors")
        else:
            print("⚠ Could not load master topics from selectors; using raw topic text")

        search_btn = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, "//input[@type='submit' and @value='Search']"))
        )
        driver.execute_script("arguments[0].click();", search_btn)
        print("✓ Clicked Search button")
        time.sleep(2)

        try:
            show_all_btn = driver.find_element(By.ID, "btnShowAllHistory")
            driver.execute_script("arguments[0].click();", show_all_btn)
            time.sleep(1)
        except Exception:
            pass

        page_source = driver.page_source
        soup = BeautifulSoup(page_source, 'html.parser')
        results_container_id = "dnn_ctr15754_StateNetDB_linkList" if USE_ARCHIVE else "dnn_ctr31026_StateNetDB_linkList"
        results_container = soup.find('div', {'id': results_container_id})

        if not results_container:
            print("✗ Results container not found")
            return pd.DataFrame()

        bill_links = results_container.find_all('a', href=lambda x: x and 'ID:bill:' in x)
        print(f"✓ Found {len(bill_links)} bills")

        for bill_link in bill_links:
            try:
                bill_number = bill_link.get_text(strip=True)
                bill_url = bill_link.get('href', '')

                def belongs_to_this_bill(field_element):
                    if not field_element:
                        return False
                    closest_bill = field_element.find_previous('a', href=lambda x: x and 'ID:bill:' in x)
                    return closest_bill == bill_link

                bill_year = "N/A"
                try:
                    text_after_link = bill_link.find_next_sibling(string=True)
                    if text_after_link:
                        text_clean = text_after_link.strip()
                        if text_clean.isdigit() and len(text_clean) == 4:
                            bill_year = text_clean
                except Exception:
                    pass

                bill_type = "N/A"
                try:
                    type_div = bill_link.find_next('div', {'style': lambda x: x and 'font-weight: bold' in x})
                    if type_div and belongs_to_this_bill(type_div):
                        bill_type = type_div.get_text(strip=True)
                except Exception:
                    pass

                status = "N/A"
                try:
                    status_b = bill_link.find_next('b', string=lambda s: s and 'Status:' in s)
                    if status_b and belongs_to_this_bill(status_b):
                        next_elem = status_b.next_sibling
                        while next_elem and isinstance(next_elem, str) and not next_elem.strip():
                            next_elem = next_elem.next_sibling
                        if next_elem:
                            status = next_elem.strip() if isinstance(next_elem, str) else next_elem.get_text(strip=True)
                except Exception:
                    pass

                date = "N/A"
                try:
                    date_b = bill_link.find_next('b', string=lambda s: s and 'Date of Last Action:' in s)
                    if date_b and belongs_to_this_bill(date_b):
                        next_elem = date_b.next_sibling
                        while next_elem and isinstance(next_elem, str) and not next_elem.strip():
                            next_elem = next_elem.next_sibling
                        if next_elem:
                            date = next_elem.strip() if isinstance(next_elem, str) else next_elem.get_text(strip=True)
                except Exception:
                    pass

                author = "N/A"
                try:
                    author_b = bill_link.find_next('b', string=lambda s: s and 'Author:' in s)
                    if author_b and belongs_to_this_bill(author_b):
                        next_elem = author_b.next_sibling
                        while next_elem and isinstance(next_elem, str) and not next_elem.strip():
                            next_elem = next_elem.next_sibling
                        if next_elem:
                            author = next_elem.strip() if isinstance(next_elem, str) else next_elem.get_text(strip=True)
                except Exception:
                    pass

                topics = "N/A"
                try:
                    topics_b = bill_link.find_next('b', string=lambda s: s and 'Topics:' in s)
                    if topics_b and belongs_to_this_bill(topics_b):
                        next_elem = topics_b.next_sibling
                        while next_elem and isinstance(next_elem, str) and not next_elem.strip():
                            next_elem = next_elem.next_sibling
                        if next_elem:
                            raw_topics = next_elem.strip() if isinstance(next_elem, str) else next_elem.get_text(' ', strip=True)
                            topics = parse_bill_topics(raw_topics, master_topics)
                except Exception:
                    pass

                associated_bills = ""
                try:
                    assoc_b = bill_link.find_next('b', string=lambda s: s and 'Associated Bills:' in s)
                    if assoc_b and belongs_to_this_bill(assoc_b):
                        next_elem = assoc_b.next_sibling
                        while next_elem and isinstance(next_elem, str) and not next_elem.strip():
                            next_elem = next_elem.next_sibling
                        if next_elem:
                            associated_bills = next_elem.strip() if isinstance(next_elem, str) else next_elem.get_text(strip=True)
                except Exception:
                    pass

                summary = "N/A"
                try:
                    summary_b = bill_link.find_next('b', string=lambda s: s and 'Summary:' in s)
                    if summary_b and belongs_to_this_bill(summary_b):
                        next_elem = summary_b.next_sibling
                        while next_elem and isinstance(next_elem, str) and not next_elem.strip():
                            next_elem = next_elem.next_sibling
                        if next_elem:
                            summary = next_elem.strip() if isinstance(next_elem, str) else next_elem.get_text(strip=True)
                except Exception:
                    pass

                history = "N/A"
                try:
                    history_list_div = bill_link.find_next('div', {'class': 'historyList'})
                    if history_list_div and belongs_to_this_bill(history_list_div):
                        entries = []
                        for br in history_list_div.find_all('br'):
                            if br.previous_sibling:
                                text = str(br.previous_sibling).strip()
                                if text:
                                    entries.append(text)

                        for content in history_list_div.find_all(string=True):
                            text = content.strip()
                            if text and text not in entries:
                                entries.append(text)

                        if entries:
                            history = '\n'.join(entries)
                except Exception:
                    pass

                bills_data.append({
                    'Bill Number': bill_number,
                    'Year': bill_year,
                    'Type': bill_type,
                    'Status': status,
                    'Last Action Date': date,
                    'Author': author,
                    'Topics': topics,
                    'Associated Bills': associated_bills,
                    'Summary': summary,
                    'History': history,
                    'URL': bill_url
                })

            except Exception as e:
                print(f"⚠ Error processing bill: {str(e)}")
                continue

        if bills_data:
            df = pd.DataFrame(bills_data)
            df['Bill Number'] = df['Bill Number'].str.strip().str.replace(r'\s+', ' ', regex=True)

            if SELECTED_STATE == "New Jersey":
                df['Bill Number'] = df['Bill Number'].str.replace(r'^NJ\s*', '', regex=True).str.replace(r'\s+', '', regex=True)

            if 'Associated Bills' in df.columns:
                df['clean_associated_bill_number'] = (
                    df['Associated Bills']
                    .fillna('')
                    .astype(str)
                    .apply(lambda value: value.split('-', 1)[0])
                    .str.replace(r'^NJ\s*', '', regex=True)
                    .str.replace(r'\s+', '', regex=True)
                )
                df.loc[df['Associated Bills'].fillna('').str.strip().eq(''), 'clean_associated_bill_number'] = ''

            df['bill_key'] = df['Bill Number'] + "_" + df['Year'].astype(str)
            df = df[['bill_key'] + [col for col in df.columns if col != 'bill_key']]

            print(f"✓ Extracted {len(df)} bills successfully")
            return df

        print("✗ No bills extracted")
        return pd.DataFrame()

    finally:
        driver.quit()


def main():
    """Run scraper and return combined DataFrame for notebook use."""
    print("="*80)
    print("NCSL SCRAPER - HEADED DEMO")
    print("="*80)

    validate_config()

    if len(SELECTED_YEARS) > 1:
        all_dfs = []
        for year in SELECTED_YEARS:
            df = scrape_bills(year=year)
            if not df.empty:
                all_dfs.append(df)
            time.sleep(1)

        if all_dfs:
            combined_df = pd.concat(all_dfs, ignore_index=True)
            print(f"\n✓ Combined {len(combined_df)} bills from {len(SELECTED_YEARS)} years")
            return combined_df
        return pd.DataFrame()

    return scrape_bills()


ncsl_df = main()
ncsl_df.head(20) if not ncsl_df.empty else ncsl_df

NCSL SCRAPER - HEADED DEMO
✓ Configuration: New Jersey - Years: 2026 (Current Database)

SCRAPING BILLS FROM NCSL CURRENT DATABASE - YEAR 2026
Navigating to https://www.ncsl.org/elections-and-campaigns/state-election-legislation-database
✓ Selected New Jersey
✓ Selected year 2026
✓ Loaded 59 master topics from selectors
✓ Clicked Search button
✓ Found 203 bills
✓ Extracted 203 bills successfully


,bill_key,Bill Number,Year,Type,Status,Last Action Date,Author,Topics,Associated Bills,Summary,History,URL,clean_associated_bill_number
0,SCR23_2026,SCR23,2026,Constitutional Amendment,"Pending - Senate State Government, Wagering, T...",1/13/2026,Singer (R),Absentee Voting - Application and Request for;...,,Proposes constitutional amendment to provide r...,01/13/2026 - FILED.\n01/13/2026 - INTRODUCED.\...,http://custom.statenet.com/public/resources.cg...,
1,ACR40_2026,ACR40,2026,Congressional Resolution,Pending - Assembly State and Local Government ...,1/13/2026,Murphy (D),Miscellaneous,,Urges United States Congress to enact Alice Pa...,01/13/2026 - FILED.\n01/13/2026 - INTRODUCED.\...,http://custom.statenet.com/public/resources.cg...,
2,AR61_2026,AR61,2026,Comprehensive Voting Rights Legislation,Pending - Assembly State and Local Government ...,1/13/2026,Park (D),Miscellaneous,,Urges US Congress adopt comprehensive voting r...,01/13/2026 - FILED.\n01/13/2026 - INTRODUCED.\...,http://custom.statenet.com/public/resources.cg...,
3,ACR65_2026,ACR65,2026,Constitutional Amendment,Pending - Assembly State and Local Government ...,1/13/2026,Kean S (R),Absentee Voting - Application and Request for;...,,Proposes constitutional amendment to provide r...,01/13/2026 - FILED.\n01/13/2026 - INTRODUCED.\...,http://custom.statenet.com/public/resources.cg...,
4,S68_2026,S68,2026,Registered Voters Photo Identification,"Pending - Senate State Government, Wagering, T...",1/13/2026,Holzapfel (R),Voter Identification,NJ A 668 - Identical,Requires registered voters to present photo ID...,01/13/2026 - FILED.\n01/13/2026 - INTRODUCED.\...,http://custom.statenet.com/public/resources.cg...,A668
5,SCR70_2026,SCR70,2026,Constitutional Amendment,"Pending - Senate State Government, Wagering, T...",1/28/2026,Amato (R),Candidates - Qualification and Running for Off...,,Proposes constitutional amendment to prohibit ...,01/13/2026 - FILED.\n01/28/2026 - INTRODUCED.\...,http://custom.statenet.com/public/resources.cg...,
6,SR74_2026,SR74,2026,Congressional Resolution,"Pending - Senate State Government, Wagering, T...",2/19/2026,Greenstein (D),Dates of Elections and Election Holidays; Misc...,,Urges Congress to make Election Day federal ho...,02/12/2026 - FILED.\n02/19/2026 - INTRODUCED.\...,http://custom.statenet.com/public/resources.cg...,
7,SJR87_2026,SJR87,2026,Safeguard American Voter Eligibility Act,"Pending - Senate State Government, Wagering, T...",2/12/2026,Holzapfel (R),Miscellaneous; Voters - Eligibility and Citize...,,Urges U.S. Senate to pass federal Safeguard Am...,02/09/2026 - FILED.\n02/12/2026 - INTRODUCED.\...,http://custom.statenet.com/public/resources.cg...,
8,A114_2026,A114,2026,Election Candidates Rules and Regulations,Pending - Assembly State and Local Government ...,1/13/2026,Barlas (R),Ballot Access for Candidates,,Allows candidates to file form attesting to th...,01/13/2026 - FILED.\n01/13/2026 - INTRODUCED.\...,http://custom.statenet.com/public/resources.cg...,
9,A115_2026,A115,2026,Blank Nominating Petition Forms,Pending - Assembly State and Local Government ...,1/13/2026,Barlas (R),Ballot Access for Candidates,,Requires blank nominating petition forms for p...,01/13/2026 - FILED.\n01/13/2026 - INTRODUCED.\...,http://custom.statenet.com/public/resources.cg...,


## General tips for web scraping
* Do spot checks and make sure that the number of items collected by your scraper matches the number of items on the webpage. Often there are issues due to pagination (ie, you only scraped the first page of results) or because the scraper is not correctly identifying the HTML elements that contain the data.
* LLMs are continually getting better at scraping, but they can get lost in the weeds of HTML. If you find that your LLM is struggling to scrape the right information, try to simplify the HTML you are sharing with it. You can do this by copying only the relevant sections of the HTML and explaining in clear terms which element you want it to collect.
* If you are scraping a lot of data, be mindful of the website's terms of service and consider implementing rate limiting (ie, adding delays between requests) in your scraper to avoid making too many requests in a short period of time.